This notebook reads every export in `data/raw/search_exports/`, merges duplicate records, and writes `data/processed/deduplicated_records.csv`. It also writes the identification counts to `data/processed/screening_counts.yaml`, which [02 Screening](02-screening.ipynb) extends.

## Method

Each export is parsed into title, abstract, authors, year, journal, and DOI. Two records are duplicates when they share a normalized DOI, or when they share a normalized title and a publication year. Matches are transitive, so a record without a DOI joins the group of a record with one when their titles and years agree. The DOI is lowercased and stripped of any resolver prefix. The title is reduced to lowercase letters and digits after accents are removed. From each group the record with the longest abstract is kept. The `record_id` is a hash of the smallest DOI or title key in the group, so it stays the same when the same exports are processed again.

The merge is written in Python with pandas and no other deduplication software. The table at the end lists merged groups for a spot check.

In [1]:
import hashlib
import re
import unicodedata
from pathlib import Path

import pandas as pd
import rispy
import yaml

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
RAW = ROOT / "data" / "raw" / "search_exports"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

NAME = re.compile(r"^(?P<database>.+?)_(?P<date>\d{4}-\d{2}-\d{2})(?:_(?P<block>[A-Za-z0-9]+))?$")
files = sorted(p for p in RAW.glob("*") if p.suffix.lower() in {".ris", ".csv"})
have_data = bool(files)
if have_data:
    print(f"{len(files)} export file(s) found.")
else:
    print("No exports found in data/raw/search_exports/. Add exports and render this notebook again.")

No exports found in data/raw/search_exports/. Add exports and render this notebook again.


In [2]:
FIELDS = ["title", "abstract", "authors", "year", "journal", "doi"]
CSV_COLUMNS = {
    "title": ["title", "article title", "document title"],
    "abstract": ["abstract"],
    "authors": ["authors", "author full names", "author(s)", "author"],
    "year": ["year", "publication year", "py"],
    "journal": ["journal", "source title", "journal/book", "publication title"],
    "doi": ["doi"],
}
RIS_KEYS = {
    "title": ["primary_title", "title", "translated_title"],
    "abstract": ["abstract", "notes_abstract"],
    "authors": ["authors", "first_authors"],
    "year": ["year", "publication_year", "date"],
    "journal": ["secondary_title", "journal_name", "alternate_title3", "alternate_title2"],
    "doi": ["doi"],
}


def first(record, keys):
    for key in keys:
        value = record.get(key)
        if value:
            return "; ".join(map(str, value)) if isinstance(value, list) else str(value)
    return ""


def read_export(path):
    if path.suffix.lower() == ".ris":
        with open(path, encoding="utf-8-sig") as fh:
            rows = [{f: first(r, RIS_KEYS[f]) for f in FIELDS} for r in rispy.load(fh)]
        return pd.DataFrame(rows, columns=FIELDS)
    df = pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")
    lower = {c.lower().strip(): c for c in df.columns}
    out = pd.DataFrame(index=df.index)
    for f in FIELDS:
        col = next((lower[c] for c in CSV_COLUMNS[f] if c in lower), None)
        out[f] = df[col] if col else ""
    return out

In [3]:
if have_data:
    frames = []
    for p in files:
        m = NAME.match(p.stem)
        if not m:
            raise ValueError(f"{p.name}: name the file <database>_<YYYY-MM-DD>[_<block>], for example scopus_2026-11-02_A.ris")
        df = read_export(p)
        df["database"] = m["database"]
        df["search_date"] = m["date"]
        df["block"] = m["block"] or ""
        df["file"] = p.name
        frames.append(df)
    records = pd.concat(frames, ignore_index=True).fillna("").astype(str)


def clean_doi(x):
    x = x.strip().lower()
    return re.sub(r"^(https?://(dx\.)?doi\.org/|doi:\s*)", "", x)


def title_key(x):
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "", x)


def year4(x):
    m = re.search(r"\d{4}", x)
    return m.group(0) if m else ""

In [4]:
if have_data:
    records["doi"] = records["doi"].map(clean_doi)
    records["year"] = records["year"].map(year4)
    records["tkey"] = records["title"].map(title_key)
    records["abstract_len"] = records["abstract"].str.len()

    parent = list(range(len(records)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    seen = {}
    for i, r in enumerate(records.itertuples()):
        keys = []
        if r.doi:
            keys.append(("doi", r.doi))
        if r.tkey and r.year:
            keys.append(("title", r.tkey, r.year))
        for k in keys:
            if k in seen:
                parent[find(i)] = find(seen[k])
            else:
                seen[k] = i
    records["group"] = [find(i) for i in range(len(records))]

    def group_key(g):
        keys = {f"doi:{d}" for d in g["doi"] if d}
        keys |= {f"title:{t}:{y}" for t, y in zip(g["tkey"], g["year"]) if t and y}
        return min(keys) if keys else f"row:{g['file'].iloc[0]}:{g.index[0]}"

    rows = []
    for _, g in records.groupby("group", sort=False):
        best = g.sort_values("abstract_len", ascending=False).iloc[0]
        rows.append({
            "record_id": "R" + hashlib.sha1(group_key(g).encode()).hexdigest()[:10],
            "title": best["title"],
            "abstract": best["abstract"],
            "authors": best["authors"],
            "year": best["year"],
            "journal": best["journal"],
            "doi": next((d for d in g["doi"] if d), ""),
            "databases": "; ".join(sorted(set(g["database"]))),
            "blocks": "; ".join(sorted(b for b in set(g["block"]) if b)),
            "n_exports": len(g),
        })
    dedup = pd.DataFrame(rows).sort_values("record_id").reset_index(drop=True)
    dedup.to_csv(OUT / "deduplicated_records.csv", index=False)

## Records per search

In [5]:
if have_data:
    by_export = (
        records.groupby(["database", "block", "search_date"], as_index=False)
        .size()
        .rename(columns={"size": "records"})
    )
    display(by_export)
    total, unique = len(records), len(dedup)
    print(f"Records identified: {total}")
    print(f"Duplicate records removed: {total - unique}")
    print(f"Unique records: {unique}")

In [6]:
if have_data:
    path = OUT / "screening_counts.yaml"
    counts = (yaml.safe_load(path.read_text()) or {}) if path.exists() else {}
    previous = counts.get("identification", {})
    is_other = by_export["database"].str.startswith("other-")

    def entry(r, strip=False):
        name = r.database.removeprefix("other-") if strip else r.database
        return {"name": name, "block": r.block or None, "search_date": r.search_date, "n": int(r.records)}

    counts["identification"] = {
        "databases": [entry(r) for r in by_export[~is_other].itertuples()],
        "other_methods": [entry(r, strip=True) for r in by_export[is_other].itertuples()],
        "duplicates_removed": int(total - unique),
        "marked_ineligible_by_automation": previous.get("marked_ineligible_by_automation", 0),
        "removed_other_reasons": previous.get("removed_other_reasons", 0),
    }
    path.write_text(yaml.safe_dump(counts, sort_keys=False))
    print("Wrote data/processed/screening_counts.yaml")

## Merged records, for a spot check

In [7]:
if have_data:
    merged = dedup[dedup["n_exports"] > 1][["record_id", "title", "year", "databases", "n_exports"]]
    if merged.empty:
        print("No records were merged.")
    else:
        display(merged.head(25))